In [1]:
import osmnx as ox

import pandas as pd
import numpy as np

from pathlib import Path

import requests
import json

from rapidfuzz import fuzz
from unidecode import unidecode

# Load các street type

In [2]:
streets_df = pd.read_csv("../data/raw/streets.csv")
streets_types = streets_df["type"].unique().tolist()
streets_types.remove("unclassified")
print(streets_types)

['trunk', 'tertiary', 'secondary', 'primary_link', 'primary', 'trunk_link', 'tertiary_link', 'secondary_link', 'motorway_link', 'motorway']


# Load graph

In [3]:
with open("../data/raw/osm_train_2019_01_03_simplified.json", "r", encoding="utf-8") as f:
    osm_data = json.load(f)

In [4]:
osm_data.keys()

dict_keys(['version', 'generator', 'osm3s', 'elements'])

In [5]:
osm_data["version"]

0.6

In [6]:
osm_data["osm3s"]

{'timestamp_osm_base': '2026-05-26T13:46:15Z',
 'copyright': 'The data included in this document is from www.openstreetmap.org. The data is made available under ODbL.'}

In [7]:
osm_elements = osm_data["elements"]

In [8]:
osm_elements_df = pd.json_normalize(osm_elements)
osm_elements_df.head()

,type,id,lat,lon,tags.highway,tags.name,tags.railway,tags.traffic_signals,tags.barrier,tags.junction,...,tags.bicycle:forward,tags.ele,tags.addr:street:name,tags.maxheight,tags.stop,tags.minspeed,tags.lay,tags.information,tags.ferry,tags.footway
0,node,91665277,10.951047,106.871543,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,node,91665279,10.943456,106.868393,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,node,91665280,10.936719,106.868170,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,node,91665282,10.932136,106.868172,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,node,91665283,10.928271,106.868053,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
osm_elements_df["type"].unique()

array(['node', 'way', 'relation'], dtype=object)

In [10]:
osm_elements_df.columns

Index(['type', 'id', 'lat', 'lon', 'tags.highway', 'tags.name', 'tags.railway',
       'tags.traffic_signals', 'tags.barrier', 'tags.junction',
       ...
       'tags.bicycle:forward', 'tags.ele', 'tags.addr:street:name',
       'tags.maxheight', 'tags.stop', 'tags.minspeed', 'tags.lay',
       'tags.information', 'tags.ferry', 'tags.footway'],
      dtype='object', length=165)

In [11]:
non_tags_attrs = [x for x in osm_elements_df.columns if not x.startswith("tags")]
non_tags_attrs

['type', 'id', 'lat', 'lon', 'nodes', 'members']

# OSM Node

In [12]:
osm_nodes_full_df = osm_elements_df[
    osm_elements_df["type"] == "node"
].dropna(axis=1, how="all")

print(osm_nodes_full_df.shape)
osm_nodes_full_df.head()

(63732, 80)


,type,id,lat,lon,tags.highway,tags.name,tags.railway,tags.traffic_signals,tags.barrier,tags.junction,...,tags.entrance,tags.noexit,tags.cuisine,tags.diet:vegetarian,tags.diet:vegan,tags.leisure,tags.addr:housename,tags.parking,tags.information,tags.ferry
0,node,91665277,10.951047,106.871543,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,node,91665279,10.943456,106.868393,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,node,91665280,10.936719,106.868170,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,node,91665282,10.932136,106.868172,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,node,91665283,10.928271,106.868053,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
(osm_nodes_full_df.isnull().sum() * 100 / len(osm_nodes_full_df)).sort_values(ascending=True)

type                     0.000000
id                       0.000000
lat                      0.000000
lon                      0.000000
tags.highway            97.971192
                          ...    
tags.diet:vegetarian    99.998431
tags.leisure            99.998431
tags.diet:vegan         99.998431
tags.parking            99.998431
tags.information        99.998431
Length: 80, dtype: float64

In [14]:
osm_nodes_df = osm_elements_df[osm_elements_df["type"] == "node"][non_tags_attrs]
print(osm_nodes_df.shape)
osm_nodes_df.head()

(63732, 6)


,type,id,lat,lon,nodes,members
0,node,91665277,10.951047,106.871543,NaN,NaN
1,node,91665279,10.943456,106.868393,NaN,NaN
2,node,91665280,10.936719,106.868170,NaN,NaN
3,node,91665282,10.932136,106.868172,NaN,NaN
4,node,91665283,10.928271,106.868053,NaN,NaN


In [15]:
osm_nodes_df.isnull().sum() * 100 / len(osm_nodes_df)

type         0.0
id           0.0
lat          0.0
lon          0.0
nodes      100.0
members    100.0
dtype: float64

In [16]:
osm_nodes_df = osm_nodes_df.drop(columns=["nodes", "members"])
osm_nodes_df.head()

,type,id,lat,lon
0,node,91665277,10.951047,106.871543
1,node,91665279,10.943456,106.868393
2,node,91665280,10.936719,106.868170
3,node,91665282,10.932136,106.868172
4,node,91665283,10.928271,106.868053


In [17]:
node_tags = sorted([x for x in osm_nodes_full_df.columns if x.startswith("tags.")])
left_tags = node_tags.copy()
print(len(node_tags))
print(node_tags)

76
['tags.access', 'tags.addr:city', 'tags.addr:district', 'tags.addr:housename', 'tags.addr:housenumber', 'tags.addr:place', 'tags.addr:postcode', 'tags.addr:province', 'tags.addr:street', 'tags.addr:subdistrict', 'tags.amenity', 'tags.barrier', 'tags.bench', 'tags.bicycle', 'tags.building', 'tags.bus', 'tags.button_operated', 'tags.crossing', 'tags.crossing_ref', 'tags.cuisine', 'tags.description', 'tags.diet:vegan', 'tags.diet:vegetarian', 'tags.email', 'tags.entrance', 'tags.fee', 'tags.ferry', 'tags.fixme', 'tags.foot', 'tags.highway', 'tags.highway_1', 'tags.information', 'tags.int_name', 'tags.internet_access', 'tags.internet_access:fee', 'tags.junction', 'tags.layer', 'tags.leisure', 'tags.level', 'tags.motorcar', 'tags.motorcycle', 'tags.mountain_pass', 'tags.name', 'tags.name:de', 'tags.name:en', 'tags.name:es', 'tags.name:pt', 'tags.name:ru', 'tags.name:vi', 'tags.network', 'tags.noexit', 'tags.note', 'tags.operator', 'tags.parking', 'tags.payment:cards', 'tags.payment:debit

In [18]:
node_addr_tags = [x for x in node_tags if x.startswith("tags.addr")]
left_tags = [x for x in left_tags if x not in node_addr_tags]
print(len(node_addr_tags))
print(len(left_tags))
print(left_tags)

9
67
['tags.access', 'tags.amenity', 'tags.barrier', 'tags.bench', 'tags.bicycle', 'tags.building', 'tags.bus', 'tags.button_operated', 'tags.crossing', 'tags.crossing_ref', 'tags.cuisine', 'tags.description', 'tags.diet:vegan', 'tags.diet:vegetarian', 'tags.email', 'tags.entrance', 'tags.fee', 'tags.ferry', 'tags.fixme', 'tags.foot', 'tags.highway', 'tags.highway_1', 'tags.information', 'tags.int_name', 'tags.internet_access', 'tags.internet_access:fee', 'tags.junction', 'tags.layer', 'tags.leisure', 'tags.level', 'tags.motorcar', 'tags.motorcycle', 'tags.mountain_pass', 'tags.name', 'tags.name:de', 'tags.name:en', 'tags.name:es', 'tags.name:pt', 'tags.name:ru', 'tags.name:vi', 'tags.network', 'tags.noexit', 'tags.note', 'tags.operator', 'tags.parking', 'tags.payment:cards', 'tags.payment:debit_cards', 'tags.payment:mastercard', 'tags.payment:telephone_cards', 'tags.payment:visa', 'tags.phone', 'tags.public_transport', 'tags.railway', 'tags.ref', 'tags.shelter', 'tags.shop', 'tags.smo

In [19]:
node_name_tags = [x for x in node_tags if x.startswith("tags.name")]
left_tags = [x for x in left_tags if x not in node_name_tags]
print(len(node_name_tags))
print(len(left_tags))
print(left_tags)

7
60
['tags.access', 'tags.amenity', 'tags.barrier', 'tags.bench', 'tags.bicycle', 'tags.building', 'tags.bus', 'tags.button_operated', 'tags.crossing', 'tags.crossing_ref', 'tags.cuisine', 'tags.description', 'tags.diet:vegan', 'tags.diet:vegetarian', 'tags.email', 'tags.entrance', 'tags.fee', 'tags.ferry', 'tags.fixme', 'tags.foot', 'tags.highway', 'tags.highway_1', 'tags.information', 'tags.int_name', 'tags.internet_access', 'tags.internet_access:fee', 'tags.junction', 'tags.layer', 'tags.leisure', 'tags.level', 'tags.motorcar', 'tags.motorcycle', 'tags.mountain_pass', 'tags.network', 'tags.noexit', 'tags.note', 'tags.operator', 'tags.parking', 'tags.payment:cards', 'tags.payment:debit_cards', 'tags.payment:mastercard', 'tags.payment:telephone_cards', 'tags.payment:visa', 'tags.phone', 'tags.public_transport', 'tags.railway', 'tags.ref', 'tags.shelter', 'tags.shop', 'tags.smoking', 'tags.source', 'tags.start_date', 'tags.supervised', 'tags.tactile_paving', 'tags.tourism', 'tags.traf

In [20]:
left_tags

['tags.access',
 'tags.amenity',
 'tags.barrier',
 'tags.bench',
 'tags.bicycle',
 'tags.building',
 'tags.bus',
 'tags.button_operated',
 'tags.crossing',
 'tags.crossing_ref',
 'tags.cuisine',
 'tags.description',
 'tags.diet:vegan',
 'tags.diet:vegetarian',
 'tags.email',
 'tags.entrance',
 'tags.fee',
 'tags.ferry',
 'tags.fixme',
 'tags.foot',
 'tags.highway',
 'tags.highway_1',
 'tags.information',
 'tags.int_name',
 'tags.internet_access',
 'tags.internet_access:fee',
 'tags.junction',
 'tags.layer',
 'tags.leisure',
 'tags.level',
 'tags.motorcar',
 'tags.motorcycle',
 'tags.mountain_pass',
 'tags.network',
 'tags.noexit',
 'tags.note',
 'tags.operator',
 'tags.parking',
 'tags.payment:cards',
 'tags.payment:debit_cards',
 'tags.payment:mastercard',
 'tags.payment:telephone_cards',
 'tags.payment:visa',
 'tags.phone',
 'tags.public_transport',
 'tags.railway',
 'tags.ref',
 'tags.shelter',
 'tags.shop',
 'tags.smoking',
 'tags.source',
 'tags.start_date',
 'tags.supervised',
 '

In [21]:
node_info_tags = node_name_tags + node_addr_tags + [
    "tags.website",
    "tags.ref",
    "tags.note",
    "tags.email",
    "tags.description",
    "tags.phone",
    "tags.operator",
    "tags.information",
]

In [22]:
left_tags = [x for x in left_tags if x not in node_info_tags]
print(len(left_tags))
left_tags

52


['tags.access',
 'tags.amenity',
 'tags.barrier',
 'tags.bench',
 'tags.bicycle',
 'tags.building',
 'tags.bus',
 'tags.button_operated',
 'tags.crossing',
 'tags.crossing_ref',
 'tags.cuisine',
 'tags.diet:vegan',
 'tags.diet:vegetarian',
 'tags.entrance',
 'tags.fee',
 'tags.ferry',
 'tags.fixme',
 'tags.foot',
 'tags.highway',
 'tags.highway_1',
 'tags.int_name',
 'tags.internet_access',
 'tags.internet_access:fee',
 'tags.junction',
 'tags.layer',
 'tags.leisure',
 'tags.level',
 'tags.motorcar',
 'tags.motorcycle',
 'tags.mountain_pass',
 'tags.network',
 'tags.noexit',
 'tags.parking',
 'tags.payment:cards',
 'tags.payment:debit_cards',
 'tags.payment:mastercard',
 'tags.payment:telephone_cards',
 'tags.payment:visa',
 'tags.public_transport',
 'tags.railway',
 'tags.shelter',
 'tags.shop',
 'tags.smoking',
 'tags.source',
 'tags.start_date',
 'tags.supervised',
 'tags.tactile_paving',
 'tags.tourism',
 'tags.traffic_signals',
 'tags.traffic_signals:direction',
 'tags.traffic_sig

## Connectivity

In [23]:
node_connectivity_tags = [
    "tags.railway",
    "tags.junction",
    "tags.crossing",
    "tags.highway"
]

In [24]:
print(osm_nodes_full_df["tags.junction"].unique())
print(osm_nodes_full_df[osm_nodes_full_df["tags.junction"].notna()].shape)

[nan 'yes']
(6, 80)


In [25]:
print(osm_nodes_full_df["tags.crossing"].unique())
print(osm_nodes_full_df[osm_nodes_full_df["tags.crossing"].notna()].shape)

[nan 'zebra' 'traffic_signals' 'uncontrolled']
(29, 80)


In [26]:
print(osm_nodes_full_df["tags.highway"].unique())
print(osm_nodes_full_df[osm_nodes_full_df["tags.highway"].notna()].shape)

[nan 'traffic_signals' 'bus_stop' 'crossing' 'motorway_junction' 'stop'
 'turning_circle']
(1293, 80)


In [27]:
print(osm_nodes_full_df["tags.railway"].unique())
print(osm_nodes_full_df[osm_nodes_full_df["tags.railway"].notna()].shape)

[nan 'level_crossing' 'station' 'crossing']
(31, 80)


## Public transportation

In [28]:
node_public_tags = [
    "tags.bus",
    "tags.ferry",
]

In [29]:
print(osm_nodes_full_df["tags.bus"].unique())
print(osm_nodes_full_df[osm_nodes_full_df["tags.bus"].notna()].shape)

[nan 'yes']
(7, 80)


In [30]:
osm_nodes_full_df["tags.ferry"].unique()
print(osm_nodes_full_df[osm_nodes_full_df["tags.ferry"].notna()].shape)

(2, 80)


# OSM Way

In [32]:
osm_ways_full_df = osm_elements_df[
    osm_elements_df["type"] == "way"
].dropna(axis=1, how="all")
osm_ways_full_df.to_csv("../data/preprocess/osm_ways_full.csv", index=False)
print(osm_ways_full_df.shape)
osm_ways_full_df.head()

(8429, 102)


,type,id,tags.highway,tags.name,tags.railway,tags.traffic_signals,tags.junction,tags.name:en,tags.layer,tags.addr:housenumber,...,tags.proposed,tags.bicycle:forward,tags.ele,tags.addr:street:name,tags.maxheight,tags.stop,tags.minspeed,tags.lay,tags.information,tags.footway
17334,way,26163073,trunk,Quốc Lộ 51,NaN,NaN,NaN,National Route 51,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17335,way,28369851,trunk,Quốc lộ 1A,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17336,way,28369853,trunk,Quốc lộ 1A,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17337,way,28379289,motorway,Đường cao tốc Hà Nội - Bắc Giang,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17338,way,28379290,motorway,Đường cao tốc Hà Nội - Bắc Giang,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
(osm_ways_full_df.isnull().sum() * 100 / len(osm_ways_full_df)).sort_values(ascending=True)

type                     0.000000
id                       0.000000
nodes                    0.000000
tags.highway             0.023728
tags.name               35.235497
                          ...    
tags.proposed           99.988136
tags.religion           99.988136
tags.oneway:motorcar    99.988136
tags.lay                99.988136
tags.footway            99.988136
Length: 102, dtype: float64

In [34]:
osm_ways_df = osm_elements_df[osm_elements_df["type"] == "way"][non_tags_attrs]
print(osm_ways_df.shape)
osm_ways_df.head()

(8429, 6)


,type,id,lat,lon,nodes,members
17334,way,26163073,NaN,NaN,"[1593262856, 1596621592, 1596621134, 159326402...",NaN
17335,way,28369851,NaN,NaN,"[311583734, 3441066901, 3441066939, 1724801556...",NaN
17336,way,28369853,NaN,NaN,"[311583796, 311583734]",NaN
17337,way,28379289,NaN,NaN,"[311730691, 311730692]",NaN
17338,way,28379290,NaN,NaN,"[440810974, 440810973, 440810971, 2913974706, ...",NaN


In [35]:
osm_ways_df.isnull().sum() * 100 / len(osm_ways_df)

type         0.0
id           0.0
lat        100.0
lon        100.0
nodes        0.0
members    100.0
dtype: float64

In [36]:
osm_ways_df = osm_ways_df.drop(columns=["lat", "lon", "members"])
osm_ways_df.head()

,type,id,nodes
17334,way,26163073,"[1593262856, 1596621592, 1596621134, 159326402..."
17335,way,28369851,"[311583734, 3441066901, 3441066939, 1724801556..."
17336,way,28369853,"[311583796, 311583734]"
17337,way,28379289,"[311730691, 311730692]"
17338,way,28379290,"[440810974, 440810973, 440810971, 2913974706, ..."


In [37]:
osm_way_tags = sorted([x for x in osm_ways_full_df.columns if x.startswith("tags.")])
left_tags = osm_way_tags.copy()
len(osm_way_tags)

99

In [38]:
way_name_tags = [
    x for x in osm_way_tags if 
        x.startswith("tags.name") or 
        x.startswith("tags.old_name") or
        x.startswith("tags.alt_name")
]
left_tags = [x for x in left_tags if x not in way_name_tags]
print(len(way_name_tags))
print(way_name_tags)

16
['tags.alt_name', 'tags.alt_name:en', 'tags.alt_name:vi', 'tags.name', 'tags.name:de', 'tags.name:en', 'tags.name:ja', 'tags.name:th', 'tags.name:vi', 'tags.name:vi-hani', 'tags.name:zh', 'tags.old_name', 'tags.old_name:en', 'tags.old_name:fr', 'tags.old_name:vi', 'tags.old_name:zh']


In [39]:
way_addr_tags = [x for x in osm_way_tags if x.startswith("tags.addr")]
left_tags = [x for x in left_tags if x not in way_addr_tags]
print(len(way_addr_tags))
print(way_addr_tags)

8
['tags.addr:city', 'tags.addr:district', 'tags.addr:housenumber', 'tags.addr:postcode', 'tags.addr:province', 'tags.addr:street', 'tags.addr:street:name', 'tags.addr:subdistrict']


In [42]:
way_lane_tags = [x for x in left_tags if x.startswith("tags.lane")]
left_tags = [x for x in left_tags if x not in way_lane_tags]
print(len(way_lane_tags))
print(way_lane_tags)

1
['tags.lanes']


In [43]:
way_bicycle_tags = [x for x in left_tags if x.startswith("tags.bicycle")]
left_tags = [x for x in left_tags if x not in way_bicycle_tags]
print(len(way_bicycle_tags))
print(way_bicycle_tags)

3
['tags.bicycle', 'tags.bicycle:forward', 'tags.bicycle:oneway']


In [44]:
way_maxspeed_tags = [x for x in left_tags if x.startswith("tags.maxspeed")]
left_tags = [x for x in left_tags if x not in way_maxspeed_tags]
print(len(way_maxspeed_tags))
print(way_maxspeed_tags)

2
['tags.maxspeed', 'tags.maxspeed:lanes']


In [45]:
way_oneway_tags = [x for x in left_tags if x.startswith("tags.oneway")]
left_tags = [x for x in left_tags if x not in way_oneway_tags]
print(len(way_oneway_tags))
print(way_oneway_tags)

4
['tags.oneway', 'tags.oneway:bicycle', 'tags.oneway:motorcar', 'tags.oneway:motorcycle']


In [46]:
way_maxweight_tags = [x for x in left_tags if x.startswith("tags.maxweight")]
left_tags = [x for x in left_tags if x not in way_maxweight_tags]
print(len(way_maxweight_tags))
print(way_maxweight_tags)

1
['tags.maxweight']


In [49]:
way_hgv_tags = [x for x in left_tags if x.startswith("tags.hgv")]
left_tags = [x for x in left_tags if x not in way_hgv_tags]
print(len(way_hgv_tags))
print(way_hgv_tags)

1
['tags.hgv']


In [50]:
way_bus_tags = [x for x in left_tags if x.startswith("tags.bus")]
left_tags = [x for x in left_tags if x not in way_bus_tags]
print(len(way_bus_tags))
print(way_bus_tags)

1
['tags.bus']


In [51]:
way_motorcycle_tags = [x for x in left_tags if x.startswith("tags.motorcycle")]
left_tags = [x for x in left_tags if x not in way_motorcycle_tags]
print(len(way_bus_tags))
print(way_motorcycle_tags)

1
['tags.motorcycle', 'tags.motorcycle:oneway']


In [52]:
way_motor_vehicle_tags = [x for x in left_tags if x.startswith("tags.motor_vehicle")]
left_tags = [x for x in left_tags if x not in way_motor_vehicle_tags]
print(len(way_motor_vehicle_tags))
print(way_motor_vehicle_tags)

1
['tags.motor_vehicle']


In [55]:
way_infos_tags = way_addr_tags + way_name_tags + [
    "tags.note",
    "tags.description",
    "tags.ref",
    "tags.source",
    "tags.source:maxspeed",
    "tags.int_ref",
    "tags.date",
    "tags.information"
]
left_tags = [x for x in left_tags if x not in way_infos_tags]

In [56]:
(osm_ways_full_df[left_tags].isnull().sum() * 100 / len(osm_ways_full_df)).sort_values(ascending=True)

tags.highway               0.023728
tags.service              74.730098
tags.bridge               93.913869
tags.layer                94.079962
tags.foot                 95.195160
tags.surface              96.630680
tags.motorroad            98.694982
tags.horse                98.920394
tags.junction             99.276308
tags.access               99.300036
tags.motorcar             99.383082
tags.cycleway             99.430537
tags.int_name             99.561039
tags.smoothness           99.667814
tags.construction         99.750860
tags.lit                  99.762724
tags.tunnel               99.857634
tags.width                99.881362
tags.bridge_name:en       99.916953
tags.bridge_name          99.916953
tags.comment              99.928817
tags.maxheight            99.952545
tags.bridge:name          99.952545
tags.motorcar:backward    99.952545
tags.sidewalk             99.952545
tags.area                 99.952545
tags.wheelchair           99.964409
tags.snowmobile           99

In [57]:
way_topology_tags = [
    "tags.highway"
]

In [58]:
left_tags

['tags.access',
 'tags.alt_ref',
 'tags.amenity',
 'tags.area',
 'tags.boundary',
 'tags.bridge',
 'tags.bridge:name',
 'tags.bridge:name:en',
 'tags.bridge:structure',
 'tags.bridge_name',
 'tags.bridge_name:en',
 'tags.car',
 'tags.comment',
 'tags.construction',
 'tags.covered',
 'tags.crossing',
 'tags.cycleway',
 'tags.ele',
 'tags.electrified',
 'tags.fixme',
 'tags.foot',
 'tags.footway',
 'tags.gauge',
 'tags.highway',
 'tags.horse',
 'tags.int_name',
 'tags.junction',
 'tags.lay',
 'tags.layer',
 'tags.level',
 'tags.lit',
 'tags.maxheight',
 'tags.minspeed',
 'tags.motorcar',
 'tags.motorcar:backward',
 'tags.motorcar:forward',
 'tags.motorroad',
 'tags.mountain_pass',
 'tags.proposed',
 'tags.railway',
 'tags.religion',
 'tags.service',
 'tags.sidewalk',
 'tags.ski',
 'tags.smoothness',
 'tags.snowmobile',
 'tags.stop',
 'tags.surface',
 'tags.traffic_signals',
 'tags.tunnel',
 'tags.wheelchair',
 'tags.width']

## Physical Quality

In [59]:
way_physical_tags = [
    #"tags.highway",
    "tags.surface"
]

In [60]:
print(osm_ways_full_df["tags.highway"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.highway"].notna()]))

['trunk' 'motorway' 'residential' 'secondary' 'tertiary' 'primary'
 'service' 'unclassified' 'trunk_link' 'secondary_link' 'motorway_link'
 'primary_link' 'tertiary_link' nan 'footway' 'path' 'living_street'
 'cycleway' 'road' 'pedestrian' 'track' 'proposed' 'steps' 'construction']
8427


In [61]:
print(osm_ways_full_df["tags.surface"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.surface"].notna()]))

[nan 'asphalt' 'paved' 'concrete' 'paving_stones' 'dirt' 'unpaved']
284


In [62]:
oh_physical_df = pd.get_dummies(
    osm_ways_full_df[["id"] + way_physical_tags], 
    columns=way_physical_tags,
)
oh_physical_df.head()

,id,tags.surface_asphalt,tags.surface_concrete,tags.surface_dirt,tags.surface_paved,tags.surface_paving_stones,tags.surface_unpaved
17334,26163073,False,False,False,False,False,False
17335,28369851,False,False,False,False,False,False
17336,28369853,False,False,False,False,False,False
17337,28379289,True,False,False,False,False,False
17338,28379290,True,False,False,False,False,False


## Structure

In [63]:
way_structure_tags = [
    "tags.layer",
    "tags.bridge",
    "tags.cutting",
    "tags.lanes",
    "tags.oneway"
]

In [64]:
print(osm_ways_full_df["tags.layer"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.layer"].notna()]))

[nan '2' '1' '5' '-1' '3' '4']
499


In [65]:
print(osm_ways_full_df["tags.bridge"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.bridge"].notna()]))

[nan 'yes' 'viaduct']
513


In [66]:
print(osm_ways_full_df["tags.tunnel"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.tunnel"].notna()]))

[nan 'yes' 'building_passage']
12


In [67]:
print(osm_ways_full_df["tags.lanes"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.lanes"].notna()]))

['4' nan '2' '6' '3' '1' '5' '8']
281


In [68]:
print(osm_ways_full_df["tags.oneway"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.oneway"].notna()]))

['yes' nan 'no' '-1' '-1;yes' 'no;yes']
2687


In [69]:
print(osm_ways_full_df["tags.oneway"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.oneway"] == "no;yes"]))

['yes' nan 'no' '-1' '-1;yes' 'no;yes']
3


In [ ]:
print(osm_ways_full_df["tags.cutting"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.cutting"].notna()]))

In [ ]:
print(osm_ways_full_df["tags.narrow"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.narrow"].notna()]))

## Control & Restriction

In [ ]:
way_control_tags = [
    "tags.maxspeed",
    "tags.minspeed",
    "tags.maxweight",
    "tags.motorroad"
]

In [ ]:
print(osm_ways_full_df["tags.maxspeed"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.maxspeed"].notna()]))

In [ ]:
print(osm_ways_full_df["tags.minspeed"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.minspeed"].notna()]))

In [ ]:
print(osm_ways_full_df["tags.maxweight"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.maxweight"].notna()]))

In [ ]:
print(osm_ways_full_df["tags.hgv"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.hgv"].notna()]))

In [ ]:
print(osm_ways_full_df["tags.motorcar"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.motorcar"].notna()]))

In [ ]:
print(osm_ways_full_df["tags.overtaking"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.overtaking"].notna()]))

In [ ]:
print(osm_ways_full_df["tags.stop"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.stop"].notna()]))

In [ ]:
print(osm_ways_full_df["tags.maxspeed:bus"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.maxspeed:bus"].notna()]))

In [ ]:
print(osm_ways_full_df["tags.maxspeed:hgv"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.maxspeed:hgv"].notna()]))

In [ ]:
print(osm_ways_full_df["tags.maxweight:hgv"].unique())
print(len(osm_ways_full_df[osm_ways_full_df["tags.maxweight:hgv"].notna()]))

# OSM Relation

In [ ]:
osm_relation_df = osm_elements_df[osm_elements_df["type"] == "relation"][non_tags_attrs]
print(osm_relation_df.shape)
osm_relation_df.head()

In [ ]:
osm_relation_df.isnull().sum() * 100 / len(osm_relation_df)

In [ ]:
osm_relation_df = osm_relation_df.drop(columns=["lat", "lon", "nodes"])
osm_relation_df.head()

In [ ]:
# Tách mỗi member thành một dòng
df_exploded = osm_relation_df.explode('members').reset_index(drop=True)

# Tách dict trong members thành nhiều cột
member_df = pd.json_normalize(df_exploded['members'])

# Kếtmember_dfhợp lại với thông tin relation
member_df = pd.concat([
    df_exploded[['type', 'id']].reset_index(drop=True), 
    member_df
], axis=1)

member_df.head()

In [ ]:
member_group = member_df.groupby("id")

In [ ]:
for count, (idx, group) in enumerate(member_group):
    if count == 9:
        break
    if len(group) > 2:
        print(group.to_string())
        print("_"* 50)

In [ ]:
member_df["role"].unique()

In [ ]:
osm_nodes_full_df[osm_nodes_full_df ["id"] == 2212447332]

In [ ]:
member_df.to_csv("../data/raw/osm_relation_2019_01_03", index=False)